In [1]:
import os
import pandas as pd
import numpy as np
import pefile
from tqdm import tqdm

In [2]:
MALWARE_DIR = "../malware_samples"

files = os.listdir(MALWARE_DIR)
len(files)

40

In [3]:
files[:10]

['PL98_BD8B082B7711BC980252F988BB0CA936',
 '650A6FCA433EE243391E4B4C11F09438',
 'B07322743778B5868475DBE66EEDAC4F',
 '6FAA4740F99408D4D2DDDD0B09BBDEFD',
 'SAM_B659D71AE168E774FAAF38DB30F4A84',
 'NV99_C9C9DBF388A8D81D8CFB4D3FC05F8E4',
 'GFT4_7DDD3D72EAD03C7518F5D47650C8572',
 'EEE99EC8AA67B05407C01094184C33D2B5A44',
 'JH78C0A33A1B472A8C16123FD696A5CE5EBB',
 'B98hX8E8622C393D7E832D39E620EAD5D3B49']

In [4]:
# Creación de función que extrae features

def extract_features(filepath):
    
    features = {}
    
    try:
        pe = pefile.PE(filepath)
        
        # número de secciones
        features["num_sections"] = len(pe.sections)
        
        # tamaño del archivo
        features["file_size"] = os.path.getsize(filepath)
        
        # número de imports
        num_imports = 0
        
        if hasattr(pe, "DIRECTORY_ENTRY_IMPORT"):
            for entry in pe.DIRECTORY_ENTRY_IMPORT:
                num_imports += len(entry.imports)
        
        features["num_imports"] = num_imports
        
    except:
        # si no se puede analizar PE
        features["num_sections"] = 0
        features["file_size"] = 0
        features["num_imports"] = 0
    
    return features

In [5]:
# Construir el dataset

data = []

for f in tqdm(files):
    
    path = os.path.join(MALWARE_DIR, f)
    
    features = extract_features(path)
    
    features["sample"] = f
    
    data.append(features)

df = pd.DataFrame(data)

100%|███████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:00<00:00, 146.06it/s]


In [6]:
# Ver el dataset

df.head()

,num_sections,file_size,num_imports,sample
0,3,344576,10,PL98_BD8B082B7711BC980252F988BB0CA936
1,3,5632,8,650A6FCA433EE243391E4B4C11F09438
2,3,5120,7,B07322743778B5868475DBE66EEDAC4F
3,3,5632,8,6FAA4740F99408D4D2DDDD0B09BBDEFD
4,3,15360,85,SAM_B659D71AE168E774FAAF38DB30F4A84


In [7]:
# Guardar el dataset

df.to_csv("../dataset/malware_dataset.csv", index=False)